# Player appearance model: tell players apart from their image

Fine-tunes a small open image model (DINOv2-small, Apache-2.0) on your hand-unified HIL-HAZ windows
(same id = same player), so track pieces can be joined after camera pans and you are asked for fewer
shirt numbers while tagging. It tests itself three ways:
- trained on half 1, tested on half 2's track pieces (and the other way round);
- trained on HILAL-HAZM, tested on another match (the night HILAL-AHLI CVAT clip);
- the untrained model as the baseline.

Results and the model go to `MyDrive/Playbook/reid_appearance/`; Claude does the analysis.

**Runtime:** L4 GPU (T4 works, slower), about 30-45 minutes. **Runtime -> Run all**; do not edit cells.
If it disconnects, Run all again: finished parts are skipped.

In [ ]:
# Cell 1 — GPU, Drive, inputs
import os, json, time, shutil, subprocess, sys
g = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True) if shutil.which('nvidia-smi') else None
if g is None or g.returncode != 0:
    raise SystemExit('No GPU. Runtime -> Change runtime type -> L4 GPU, then re-run.')
GPU = g.stdout.strip(); print('GPU:', GPU)
from google.colab import drive
drive.mount('/content/drive')
PBD = '/content/drive/MyDrive/Playbook'
if not os.path.isdir(PBD):
    raise SystemExit('MyDrive/Playbook not found. Other Google account? Add a shortcut to the shared Playbook folder in My Drive.')
DEST = f'{PBD}/reid_appearance'; os.makedirs(DEST, exist_ok=True)
HALVES = {'h1': (f'{PBD}/benchmark/HIL-HAZ_half1.mp4', 'per_frame_tracks_half1_unified.csv', (1066, 3333), f'{PBD}/reid_windows/half1/per_frame_tracks_small.csv.gz'),
          'h2': (f'{PBD}/benchmark/HIL-HAZ_half2.mp4', 'per_frame_tracks_half2_unified.csv', (4270, 5720), f'{PBD}/reid_windows/half2/per_frame_tracks_small.csv.gz')}
missing = [p for v in HALVES.values() for p in (v[0], v[3]) if not os.path.exists(p)]
if missing:
    raise SystemExit(f'Missing in Drive: {missing}')
print('inputs found')

In [ ]:
# Cell 2 — Our repo, the two CVAT clips, packages
PB, BRANCH = '/content/playbook', 'claude/setup-gpu-video-testing-JhgUH'
if not os.path.isdir(PB + '/.git'):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', '--branch', BRANCH,
                    'https://github.com/MuwafagQ/Playbook-program.git', PB], check=True)
else:
    subprocess.run(['git', 'fetch', '-q', '--depth', '1', 'origin', BRANCH], cwd=PB, check=True)
    subprocess.run(['git', 'reset', '-q', '--hard', 'FETCH_HEAD'], cwd=PB, check=True)
for m in [m for m in list(sys.modules) if m.split('.')[0] in ('vision', 'tools', 'core', 'geometry')]:
    del sys.modules[m]
os.chdir(PB); sys.path.insert(0, PB)
CLIPS = {'hazmB': ('HILAL-HAZM_match_B_up10.mp4', 'data/cvat/hilal_hazm_B'),
         'ahliB': ('HILAL-AHLI_match_B-up8.mp4', 'data/cvat/hilal_ahli_B')}
VID = '/content/videos'; os.makedirs(VID, exist_ok=True)
subprocess.run(['git', 'fetch', '-q', '--depth', '1', 'origin', 'final-deliverable'], cwd=PB, check=True)
for f, _ in CLIPS.values():
    if not os.path.exists(f'{VID}/{f}'):
        open(f'{VID}/{f}', 'wb').write(subprocess.run(['git', 'show', f'FETCH_HEAD:{f}'], cwd=PB,
                                                      capture_output=True, check=True).stdout)
!pip install -q "transformers>=4.47"
import numpy as np, pandas as pd, torch
from tools.reid_appearance import Embedder, crops_from_video, piece_means, retrieval, sample_rows, train
print('ready')

In [ ]:
# Cell 3 — Player crops: hand-labelled players (training / testing) and our track pieces
PEOPLE = (1, 2)
SETS = {}
for h, (video, gt_csv, (lo, hi), run_csv) in HALVES.items():
    gt = pd.read_csv(gt_csv, low_memory=False)
    gt = gt[(gt.frame >= lo) & (gt.frame <= hi) & gt.class_id.isin(PEOPLE)].copy()
    gt['label'] = h + ':' + gt.class_id.astype(str) + ':' + gt.display_track_id.astype(str)
    run = pd.read_csv(run_csv)
    run = run[(run.frame >= lo) & (run.frame <= hi) & run.class_id.isin(PEOPLE) & (run.raw_tracker_id >= 0)]
    t0 = time.time()
    SETS[f'gt_{h}'] = crops_from_video(video, sample_rows(gt, 'label', every=3))
    SETS[f'pc_{h}'] = crops_from_video(video, sample_rows(run, 'raw_tracker_id', per_id=12))
    print(h, {k: len(v[1]) for k, v in SETS.items() if k.endswith(h)}, f'{time.time() - t0:.0f}s')
for c, (f, d) in CLIPS.items():
    t = pd.read_csv(f'{d}/tracks.csv')
    t = t[t.class_id.isin(PEOPLE)].copy()
    t['label'] = c + ':' + t.track_id.astype(str)
    tm = json.load(open(f'{d}/teams.json'))
    team = {**{i: 0 for i in tm['team0']}, **{i: 1 for i in tm['team1']}, **{i: 2 for i in tm.get('goalkeepers', [])}}
    t['team'] = t.track_id.map(team).fillna(-1).astype(int)
    SETS[f'gt_{c}'] = crops_from_video(f'{VID}/{f}', sample_rows(t, 'label', every=3 if c == 'hazmB' else 2))
    print(c, len(SETS[f'gt_{c}'][1]))
# a contact sheet to eyeball the crops
import matplotlib.pyplot as plt
cr, rows = SETS['gt_h1']
pick = rows.groupby('label').head(1).index[:16]
plt.figure(figsize=(16, 4))
for i, j in enumerate(pick):
    plt.subplot(1, 16, i + 1); plt.imshow(cr[j]); plt.axis('off'); plt.title(rows.label[j].split(':')[-1], fontsize=8)
plt.show()

In [ ]:
# Cell 4 — Train and test (skips finished models)
RES = f'{DEST}/results.json'
results = json.load(open(RES)) if os.path.exists(RES) else {}
ahli_cr, ahli_rows = SETS['gt_ahliB']

def evaluate(name, model):
    # cross-match test + piece embeddings for both halves (saved for Claude's joining analysis)
    e = model.embed(ahli_cr)
    r = {'ahliB_all': retrieval(e, ahli_rows.label, ahli_rows.frame),
         'ahliB_same_team': retrieval(e, ahli_rows.label, ahli_rows.frame, groups=ahli_rows.team)}
    for h in HALVES:
        cr, rows = SETS[f'pc_{h}']
        np.savez_compressed(f'{DEST}/pieces_{name}_{h}.npz', emb=model.embed(cr).astype(np.float16),
                            piece=rows.raw_tracker_id.to_numpy(), frame=rows.frame.to_numpy())
        cr, rows = SETS[f'gt_{h}']
        r[f'{h}_gt'] = retrieval(model.embed(cr), rows.label, rows.frame)
    return r

PLAN = {'zero_shot': None, 'trained_h1': ['gt_h1'], 'trained_h2': ['gt_h2'],
        'trained_all': ['gt_h1', 'gt_h2', 'gt_hazmB']}
for name, train_sets in PLAN.items():
    if name in results:
        print(name, 'already done:', results[name]['ahliB_same_team']); continue
    t0 = time.time()
    model = Embedder()
    log = []
    if train_sets:
        cr = np.concatenate([SETS[s][0] for s in train_sets])
        lab = np.concatenate([SETS[s][1].label.to_numpy() for s in train_sets])
        log = train(model, cr, lab, steps=1500)
        del cr
    results[name] = {**evaluate(name, model), 'train_sets': train_sets, 'loss_log': log,
                     'minutes': round((time.time() - t0) / 60, 1), 'gpu': GPU}
    if name == 'trained_all':
        torch.save(model.state_dict(), f'{DEST}/appearance_model.pt')
    json.dump(results, open(RES, 'w'), indent=1)
    print(name, {k: v for k, v in results[name].items() if k not in ('loss_log', 'train_sets')})
    del model; torch.cuda.empty_cache()
print(f'\nDone. Tell Claude the results are in {DEST}.')